# BigNeuron Optimization 알고리즘 비교

BigNeuron GitHub에서 다운로드한 optimization 결과(`results/Optimization/`)와 우리 방법을 동일 메트릭으로 비교합니다.

| Metric | Direction | Description |
|--------|-----------|-------------|
| **F1** | ↑ | Node-level, 2µm threshold |
| **Precision** | ↑ | Auto → Gold 매칭률 |
| **Recall** | ↑ | Gold → Auto 매칭률 |
| **ESA** | ↓ | Bidirectional nearest-neighbor mean distance (µm) |
| **PDS** | ↓ | Fraction of nodes exceeding 2µm threshold |
| **av_fragmentation** | ↓ | Gold edge 중 gap 있는 비율 |
| **contr_ratio** | →1 | auto / gold contraction ratio |
| **tips_ratio** | →1 | auto_tips / gold_tips |

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')

ROOT         = Path('..').resolve()
GOLD_DIR     = ROOT / 'data' / 'gold_standard'
RESULTS_DIR  = ROOT / 'results'
OPT_DIR      = RESULTS_DIR / 'Optimization'
CACHE_CSV    = Path('optimization_results_raw.csv')
THRESHOLD_UM = 2.0

SAMPLE_MAP = {
    'neuron2':                   'neuron2',
    'neuron4':                   'neuron4',
    '1201_01_s06b_L36_Sum_ch2':  '1201_01_s06b_L36_Sum_ch2.tif',
    '1201_01_s10mm_ch2':         '1201_01_s10mm_ch2.tif',
    '080926a':  '080926a.tif',
    '091201c1':  '091201c1.tif',
    '091202c2':  '091202c2.tif',
    '091204c2':  '091204c2.tif',
    '091226c2':  '091226c2.tif',
    '100108c3':  '100108c3.tif',
    '100110c4':  '100110c4.tif',
    '110203c3':  '110203c3.tif',
    '140918c3':  '140918c3.tif',
    '140918c7':  '140918c7.tif',
    '140918c8':  '140918c8.tif',
    '140918c9':  '140918c9.tif',
    '140921c1':  '140921c1.tif',
    '140921c3':  '140921c3.tif',
    '140921c4':  '140921c4.tif',
    '140921c5':  '140921c5.tif',
    '140921c6':  '140921c6.tif',
    '140921c9':  '140921c9.tif',
    '140921c12':  '140921c12.tif',
    '140921c14':  '140921c14.tif',
    '140921c16':  '140921c16.tif',
    '140921c22':  '140921c22.tif',
}

OURS_SAMPLE_MAP = {
    'neuron2':                          'neuron2',
    'neuron4':                          'neuron4',
    '1201_01_s06b_L36_Sum_ch2.tif':     '1201_01_s06b_L36_Sum_ch2',
    '1201_01_s10mm_ch2.tif':            '1201_01_s10mm_ch2',
    '080926a.tif':  '080926a',
    '091201c1.tif':  '091201c1',
    '091202c2.tif':  '091202c2',
    '091204c2.tif':  '091204c2',
    '091226c2.tif':  '091226c2',
    '100108c3.tif':  '100108c3',
    '100110c4.tif':  '100110c4',
    '110203c3.tif':  '110203c3',
    '140918c3.tif':  '140918c3',
    '140918c7.tif':  '140918c7',
    '140918c8.tif':  '140918c8',
    '140918c9.tif':  '140918c9',
    '140921c1.tif':  '140921c1',
    '140921c3.tif':  '140921c3',
    '140921c4.tif':  '140921c4',
    '140921c5.tif':  '140921c5',
    '140921c6.tif':  '140921c6',
    '140921c9.tif':  '140921c9',
    '140921c12.tif':  '140921c12',
    '140921c14.tif':  '140921c14',
    '140921c16.tif':  '140921c16',
    '140921c22.tif':  '140921c22',
}

# ours 결과 있는 샘플만 포함
_ours_have = {p.stem for p in (RESULTS_DIR / 'ours').glob('*.swc')}
SAMPLES = [opt for opt, gold in SAMPLE_MAP.items() if gold in _ours_have]
print(f'ours 결과 있는 샘플: {len(SAMPLES)}개 / 전체 {len(SAMPLE_MAP)}개')
SAMPLE_SHORT = {s: s.replace('1201_01_','') for s in SAMPLES}

COLOR_OURS = '#e05c4b'
COLOR_OPT  = '#6baed6'

METRICS     = ['f1', 'precision', 'recall', 'esa', 'pds', 'av_fragmentation', 'contr_ratio', 'tips_ratio']
METRIC_DIR  = {'f1': '↑', 'precision': '↑', 'recall': '↑',
               'esa': '↓', 'pds': '↓', 'av_fragmentation': '↓',
               'contr_ratio': '→1', 'tips_ratio': '→1'}

METRIC_CFG = {
    'f1':               dict(cmap='YlGn',   vmin=0,    vmax=0.80, label='F1 ↑'),
    'precision':        dict(cmap='YlGn',   vmin=0,    vmax=0.90, label='Precision ↑'),
    'recall':           dict(cmap='YlGn',   vmin=0,    vmax=0.90, label='Recall ↑'),
    'esa':              dict(cmap='YlGn_r', vmin=0,    vmax=20.0, label='ESA ↓ (µm)'),
    'pds':              dict(cmap='YlGn_r', vmin=0,    vmax=0.80, label='PDS ↓'),
    'av_fragmentation': dict(cmap='YlGn_r', vmin=0,    vmax=1.00, label='Frag ↓'),
    'contr_ratio':      dict(cmap='RdYlGn', vmin=0.85, vmax=1.15, label='Contr ratio →1'),
    'tips_ratio':       dict(cmap='RdYlGn', vmin=0.5,  vmax=3.0,  label='Tips ratio →1'),
}

## Optimization 폴더 파싱

In [ ]:
def parse_opt_swcs(sample_dir):
    result = {}
    for f in sorted(sample_dir.glob('*.swc')):
        m = re.search(r'\.v3dpbd_(.+)$', f.stem)
        if m:
            result[m.group(1)] = f
    return result

opt_catalog = {}
for sample_dir in sorted(OPT_DIR.iterdir()):
    if sample_dir.is_dir() and sample_dir.name in SAMPLE_MAP:
        algos = parse_opt_swcs(sample_dir)
        opt_catalog[sample_dir.name] = algos
        print(f'{sample_dir.name}: {len(algos)} algorithms')

## Helper functions

In [ ]:
def read_pixelsize(gold_stem):
    txt = (GOLD_DIR / f'{gold_stem}.pixelsize.txt').read_text()
    m = re.search(r'Voxel size:\s*([\d.]+)x[\d.]+x([\d.]+)', txt, re.I)
    return float(m.group(1)), float(m.group(2))

def load_swc(path, scale_xy=1.0, scale_z=1.0):
    nodes = {}
    for line in Path(path).read_text().splitlines():
        if line.startswith('#') or not line.strip(): continue
        p = line.split()
        if len(p) < 7: continue
        nid = int(p[0])
        nodes[nid] = dict(x=float(p[2])*scale_xy, y=float(p[3])*scale_xy,
                          z=float(p[4])*scale_z, r=float(p[5]), pid=int(p[6]))
    return nodes

def build_children(nodes):
    ch = {nid: [] for nid in nodes}
    for nid, n in nodes.items():
        if n['pid'] in ch: ch[n['pid']].append(nid)
    return ch

def pos_array(nodes):
    ids = sorted(nodes)
    arr = np.array([[nodes[i]['x'], nodes[i]['y'], nodes[i]['z']] for i in ids], dtype=np.float32)
    return arr, ids

def count_tips(nodes, children):
    return sum(1 for nid in nodes if not children.get(nid) and nodes[nid]['pid'] != -1)

def compute_esa_pds_f1(auto_nodes, gold_nodes, threshold_um):
    a_arr, _ = pos_array(auto_nodes); g_arr, _ = pos_array(gold_nodes)
    g_tree = cKDTree(g_arr); a_tree = cKDTree(a_arr)
    d_a2g, _ = g_tree.query(a_arr); d_g2a, _ = a_tree.query(g_arr)
    esa = float((d_a2g.mean() + d_g2a.mean()) / 2)
    pds = float((d_a2g > threshold_um).sum() + (d_g2a > threshold_um).sum()) / (len(d_a2g) + len(d_g2a))
    precision = float((d_a2g <= threshold_um).mean())
    recall    = float((d_g2a <= threshold_um).mean())
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return dict(esa=esa, pds=pds, f1=f1, precision=precision, recall=recall)

def compute_fragmentation(gold_nodes, gold_children, auto_nodes, threshold_um, step_um=1.0):
    a_arr, _ = pos_array(auto_nodes); a_tree = cKDTree(a_arr)
    n_frag = n_total = 0
    for pid, kids in gold_children.items():
        if pid not in gold_nodes: continue
        pn = gold_nodes[pid]
        for cid in kids:
            if cid not in gold_nodes: continue
            cn = gold_nodes[cid]
            p0 = np.array([pn['x'], pn['y'], pn['z']]); p1 = np.array([cn['x'], cn['y'], cn['z']])
            L = float(np.linalg.norm(p1 - p0))
            if L < 1e-6: continue
            samps = p0 + np.linspace(0,1,max(2,int(L/step_um)+1))[:,None]*(p1-p0)
            d, _ = a_tree.query(samps)
            if (d > threshold_um).any(): n_frag += 1
            n_total += 1
    return n_frag / n_total if n_total else float('nan')

def compute_contraction(nodes, children):
    bifurc = {nid for nid, ch in children.items() if len(ch) >= 2}
    roots  = [nid for nid, n in nodes.items() if n['pid'] == -1]
    starts = set(roots) | bifurc | {c for nid in bifurc for c in children.get(nid, [])}
    vals = []
    for start in starts:
        if start not in nodes: continue
        cur, path_len = start, 0.0
        while True:
            kids = children.get(cur, [])
            if len(kids) != 1: break
            nxt = kids[0]
            n, p_ = nodes[nxt], nodes[cur]
            path_len += np.linalg.norm([n['x']-p_['x'], n['y']-p_['y'], n['z']-p_['z']])
            cur = nxt
        if path_len < 1e-6: continue
        sn, se = nodes[start], nodes[cur]
        vals.append(np.linalg.norm([sn['x']-se['x'], sn['y']-se['y'], sn['z']-se['z']]) / path_len)
    return float(np.mean(vals)) if vals else float('nan')

def evaluate_swc(swc_path, gold_nodes, gold_ch, vxy, vz, is_ours=False):
    auto = load_swc(swc_path) if is_ours else load_swc(swc_path, scale_xy=vxy, scale_z=vz)
    auto_ch = build_children(auto)
    m = compute_esa_pds_f1(auto, gold_nodes, THRESHOLD_UM)
    frag = compute_fragmentation(gold_nodes, gold_ch, auto, THRESHOLD_UM)
    contr_a = compute_contraction(auto, auto_ch)
    contr_g = compute_contraction(gold_nodes, gold_ch)
    a_tips = count_tips(auto, auto_ch); g_tips = count_tips(gold_nodes, gold_ch)
    return dict(
        esa=m['esa'], pds=m['pds'], f1=m['f1'], precision=m['precision'], recall=m['recall'],
        av_fragmentation=frag,
        contr_ratio=contr_a/contr_g if contr_g > 1e-6 else float('nan'),
        tips_ratio=a_tips/g_tips if g_tips > 0 else float('nan'),
        tips_auto=a_tips, tips_gold=g_tips,
    )

## Optimization 알고리즘 평가 실행

`optimization_results_raw.csv`가 있으면 캐시에서 로드합니다. 삭제하면 재계산합니다.

In [ ]:
if CACHE_CSV.exists():
    df_opt = pd.read_csv(CACHE_CSV)
    print(f'캐시 로드: {len(df_opt)} rows ({CACHE_CSV})')
else:
    rows = []
    for sample_stem, algos in opt_catalog.items():
        gold_stem = SAMPLE_MAP[sample_stem]
        vxy, vz = read_pixelsize(gold_stem)
        gold = load_swc(GOLD_DIR / f'{gold_stem}.swc', scale_xy=vxy, scale_z=vz)
        gold_ch = build_children(gold)
        print(f'\n=== {sample_stem} ===')
        for algo, swc_path in algos.items():
            try:
                res = evaluate_swc(swc_path, gold, gold_ch, vxy, vz)
                rows.append(dict(sample=sample_stem, algorithm=algo, **res))
                print(f'  {algo:55s}  F1={res["f1"]:.3f}  ESA={res["esa"]:.2f}  PDS={res["pds"]:.3f}')
            except Exception as e:
                print(f'  ERROR {algo}: {e}')
    df_opt = pd.DataFrame(rows)
    df_opt.to_csv(CACHE_CSV, index=False)
    print(f'\n저장 완료: {CACHE_CSV}')

df_opt.head()

## Ours 결과 합치기

In [ ]:
df_ours_raw = pd.read_csv('../evaluation/results_raw.csv')
df_ours_raw = df_ours_raw[df_ours_raw['method'] == 'ours'].copy()
df_ours_raw['sample'] = df_ours_raw['sample'].map(OURS_SAMPLE_MAP).fillna(df_ours_raw['sample'])
df_ours_raw['algorithm'] = 'ours'
df_ours = df_ours_raw[['sample', 'algorithm'] + METRICS].copy()

df_all = pd.concat([df_opt[['sample', 'algorithm'] + METRICS], df_ours], ignore_index=True)
df_all['sample'] = pd.Categorical(df_all['sample'], categories=SAMPLES, ordered=True)
df_all = df_all.sort_values(['sample', 'algorithm']).reset_index(drop=True)

print(f'Total: {len(df_all)} rows  (opt: {len(df_opt)}, ours: {len(df_ours)})')
df_all.groupby('sample')['algorithm'].count()

## F1 순위 테이블 (샘플별 + 평균)

In [ ]:
pivot_f1 = df_all.pivot_table(index='algorithm', columns='sample', values='f1')
pivot_f1.columns = [SAMPLE_SHORT[c] for c in pivot_f1.columns]
pivot_f1['mean'] = pivot_f1.mean(axis=1)
pivot_f1 = pivot_f1.sort_values('mean', ascending=False).round(3)

def hl_ours(row):
    return ['background-color: #ffe0dc; font-weight: bold' if row.name == 'ours' else '' for _ in row]

pivot_f1.style \
    .apply(hl_ours, axis=1) \
    .background_gradient(cmap='YlGn', vmin=0, vmax=0.8) \
    .format('{:.3f}', na_rep='-') \
    .set_caption('F1 Score (2µm threshold) — 내림차순 정렬')

## 전체 메트릭 평균 요약

In [ ]:
# 알고리즘별 샘플 평균 (F1 기준 정렬)
algo_order = pivot_f1.index.tolist()  # F1 내림차순
summary = df_all.groupby('algorithm')[METRICS].mean().loc[algo_order].round(3)

col_rename = {
    'f1': 'F1 ↑', 'precision': 'Prec ↑', 'recall': 'Rec ↑',
    'esa': 'ESA ↓', 'pds': 'PDS ↓', 'av_fragmentation': 'Frag ↓',
    'contr_ratio': 'Contr →1', 'tips_ratio': 'Tips →1',
}
summary.columns = [col_rename[c] for c in summary.columns]

def hl_ours_row(row):
    return ['background-color: #ffe0dc; font-weight: bold' if row.name == 'ours' else '' for _ in row]

summary.style \
    .apply(hl_ours_row, axis=1) \
    .background_gradient(cmap='YlGn',   subset=['F1 ↑', 'Prec ↑', 'Rec ↑'], vmin=0, vmax=0.8) \
    .background_gradient(cmap='YlGn_r', subset=['ESA ↓', 'PDS ↓', 'Frag ↓'], vmin=0, vmax=1) \
    .format('{:.3f}', na_rep='-') \
    .set_caption('샘플 평균 전체 메트릭 (F1 기준 내림차순)')

## 샘플별 순위 — F1 / ESA / PDS

In [ ]:
metric_rows = [
    ('F1 ↑',   'f1',   True,  (0, 1.0)),
    ('ESA ↓',  'esa',  False, None),
    ('PDS ↓',  'pds',  False, (0, 1.0)),
]

n_rows, n_cols = len(metric_rows), len(SAMPLES)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 8 * n_rows))

for row_i, (m_label, m_col, higher_better, xlim) in enumerate(metric_rows):
    for col_i, sample in enumerate(SAMPLES):
        ax = axes[row_i, col_i]
        sub = df_all[df_all['sample'] == sample].dropna(subset=[m_col])
        if sub.empty:
            ax.axis('off'); continue

        sub = sub.sort_values(m_col, ascending=higher_better)  # 좋은 방향이 위
        colors = [COLOR_OURS if a == 'ours' else COLOR_OPT for a in sub['algorithm']]
        bars = ax.barh(sub['algorithm'], sub[m_col], color=colors, edgecolor='white', linewidth=0.3)

        for bar, val in zip(bars, sub[m_col]):
            ax.text(bar.get_width() + bar.get_width() * 0.01 + 1e-6,
                    bar.get_y() + bar.get_height() / 2,
                    f'{val:.3f}', va='center', ha='left', fontsize=6.5)

        if xlim: ax.set_xlim(*xlim)
        ax.set_xlabel(m_label, fontsize=9)
        ax.tick_params(axis='y', labelsize=6.5)
        if row_i == 0:
            ax.set_title(SAMPLE_SHORT[sample], fontsize=11, fontweight='bold')

from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(color=COLOR_OURS, label='ours'),
    Patch(color=COLOR_OPT,  label='BigNeuron optimization'),
], loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.01))
fig.suptitle('샘플별 F1 / ESA / PDS 순위', fontsize=13, y=1.005)
plt.tight_layout()
plt.savefig('optimization_metrics_rank.png', dpi=150, bbox_inches='tight')
plt.show()

## Heatmap — 메트릭별 × 알고리즘 × 샘플

In [ ]:
# F1 평균 기준 알고리즘 정렬
algo_order_hm = pivot_f1.sort_values('mean', ascending=False).index.tolist()
n_algo = len(algo_order_hm)
sample_labels = [SAMPLE_SHORT[s] for s in SAMPLES]

metric_pairs = [
    ('f1', 'precision'),
    ('recall', 'esa'),
    ('pds', 'av_fragmentation'),
    ('contr_ratio', 'tips_ratio'),
]

fig, axes = plt.subplots(len(metric_pairs), 2,
                         figsize=(len(SAMPLES) * 2.8 * 2, max(6, n_algo * 0.38) * len(metric_pairs)))

for pair_i, (m1, m2) in enumerate(metric_pairs):
    for m_i, metric in enumerate([m1, m2]):
        ax = axes[pair_i, m_i]
        cfg = METRIC_CFG[metric]

        pivot = df_all.pivot_table(index='algorithm', columns='sample', values=metric)
        pivot = pivot.reindex(index=algo_order_hm, columns=SAMPLES)
        data = pivot.values.astype(float)

        im = ax.imshow(data, aspect='auto', cmap=cfg['cmap'], vmin=cfg['vmin'], vmax=cfg['vmax'])

        ax.set_xticks(range(len(SAMPLES)))
        ax.set_xticklabels(sample_labels, fontsize=8)
        ax.set_yticks(range(n_algo))
        yticklabels = ax.set_yticklabels(algo_order_hm, fontsize=6.5)
        if 'ours' in algo_order_hm:
            idx = algo_order_hm.index('ours')
            yticklabels[idx].set_color(COLOR_OURS)
            yticklabels[idx].set_fontweight('bold')

        for i in range(n_algo):
            for j in range(len(SAMPLES)):
                val = data[i, j]
                if not np.isnan(val):
                    fmt = f'{val:.2f}' if metric != 'esa' else f'{val:.1f}'
                    norm_val = (val - cfg['vmin']) / (cfg['vmax'] - cfg['vmin'] + 1e-9)
                    txt_color = 'white' if norm_val > 0.6 else 'black'
                    ax.text(j, i, fmt, ha='center', va='center', fontsize=5.5, color=txt_color)

        plt.colorbar(im, ax=ax, label=cfg['label'], shrink=0.7, pad=0.01)
        ax.set_title(cfg['label'], fontsize=9, fontweight='bold')

fig.suptitle('메트릭별 Heatmap — BigNeuron Optimization vs Ours (F1 기준 정렬)', fontsize=12, y=1.005)
plt.tight_layout()
plt.savefig('optimization_heatmap_all.png', dpi=150, bbox_inches='tight')
plt.show()

## Ours 백분위 — 메트릭별

In [ ]:
show_metrics = ['f1', 'precision', 'recall', 'esa', 'pds', 'av_fragmentation']
higher = {'f1', 'precision', 'recall'}  # 높을수록 좋음

rows_pct = []
for sample in SAMPLES:
    sub = df_all[df_all['sample'] == sample]
    if 'ours' not in sub['algorithm'].values: continue
    row = {'sample': SAMPLE_SHORT[sample]}
    for m in show_metrics:
        s = sub.dropna(subset=[m]).sort_values(m, ascending=(m not in higher)).reset_index(drop=True)
        if s.empty or 'ours' not in s['algorithm'].values: continue
        rank = s[s['algorithm'] == 'ours'].index[0] + 1
        n = len(s)
        ours_val = s.loc[s['algorithm'] == 'ours', m].values[0]
        pct = (n - rank) / (n - 1) * 100 if n > 1 else 100.0
        row[m] = f'{ours_val:.3f} ({rank}/{n}, 상위{pct:.0f}%)'
    rows_pct.append(row)

df_pct = pd.DataFrame(rows_pct).set_index('sample')
df_pct.columns = [col_rename[c] for c in df_pct.columns]
display(df_pct)

# 전체 평균 순위
print('\n전체 평균 순위 (모든 샘플 평균):')
for m in show_metrics:
    s = df_all.groupby('algorithm')[m].mean().dropna()
    s_sorted = s.sort_values(ascending=(m not in higher)).reset_index()
    if 'ours' not in s_sorted['algorithm'].values: continue
    rank = s_sorted[s_sorted['algorithm'] == 'ours'].index[0] + 1
    n = len(s_sorted)
    val = s['ours']
    pct = (n - rank) / (n - 1) * 100 if n > 1 else 100.0
    dir_sym = '↑' if m in higher else '↓'
    print(f'  {col_rename[m]:12s}  {val:6.3f}  {rank:2d}/{n:<2d}  상위{pct:.0f}%')

## 샘플별 전체 메트릭 상세

In [ ]:
display_cols = ['algorithm', 'f1', 'precision', 'recall', 'esa', 'pds', 'av_fragmentation', 'contr_ratio', 'tips_ratio']
fmt_cols = [c for c in display_cols if c != 'algorithm']

def hl_ours_detail(row):
    return ['background-color: #ffe0dc; font-weight: bold' if row['algorithm'] == 'ours' else '' for _ in row]

for sample in SAMPLES:
    sub = df_all[df_all['sample'] == sample].dropna(subset=['f1']) \
              .sort_values('f1', ascending=False).reset_index(drop=True)
    if sub.empty: continue
    tbl = sub[display_cols].copy()
    tbl[fmt_cols] = tbl[fmt_cols].round(3)
    print(f'\n━━━ {SAMPLE_SHORT[sample]} ━━━')
    display(
        tbl.style
           .apply(hl_ours_detail, axis=1)
           .background_gradient(cmap='YlGn',   subset=['f1', 'precision', 'recall'], vmin=0, vmax=0.8)
           .background_gradient(cmap='YlGn_r', subset=['esa', 'pds', 'av_fragmentation'], vmin=0, vmax=1)
           .format('{:.3f}', subset=fmt_cols, na_rep='-')
           .hide(axis='index')
    )